# 🌍 Notebook 5: Casos Reales de ML por Industria
### Curso: Machine Learning Avanzado — Módulo 1

**Objetivo:** Integrar todos los conceptos del módulo en casos de uso reales de diferentes industrias.

| Caso | Industria | Problema ML | Técnicas |
|------|-----------|-------------|----------|
| 1 | 💳 Fintech | Detección de fraude en tiempo real | Lists, Dicts, Sets, OOP |
| 2 | 🏥 Salud | Predicción de readmisión hospitalaria | NumPy, Broadcasting, SVD |
| 3 | 🛒 Retail | Sistema de recomendación colaborativo | Matrices, Factorización |
| 4 | 🏭 Industria | Mantenimiento predictivo de maquinaria | Generators, memmap |
| 5 | 🌱 Agro | Clasificación de enfermedades en cultivos | Pipeline OOP completo |

---
## 💳 Caso 1: Fintech — Detección de Fraude en Tiempo Real

**Contexto:** Un banco digital procesa 50,000 transacciones/minuto y necesita clasificar cada una en menos de 5ms.

**Técnicas:** `dataclass` con `__slots__`, `deque` para ventana deslizante, scoring vectorizado con NumPy.

In [ ]:
import numpy as np
import time
from collections import deque
from dataclasses import dataclass
from typing import Dict, List, Tuple


class Transaccion:
    __slots__ = ['id', 'usuario_id', 'monto', 'pais',
                 'categoria', 'hora', 'es_online', 'timestamp']

    def __init__(self, id, usuario_id, monto, pais,
                 categoria, hora, es_online, timestamp):
        self.id = id
        self.usuario_id = usuario_id
        self.monto = monto
        self.pais = pais
        self.categoria = categoria
        self.hora = hora
        self.es_online = es_online
        self.timestamp = timestamp


class PerfilUsuario:
    """Mantiene estadisticas en linea por usuario para feature engineering."""
    __slots__ = ['usuario_id', '_montos', '_paises', '_categorias',
                 '_horas', '_n_transacciones', '_ultima_transaccion']

    def __init__(self, usuario_id: str, ventana: int = 30):
        self.usuario_id = usuario_id
        self._montos = deque(maxlen=ventana)
        self._paises = deque(maxlen=ventana)
        self._categorias = deque(maxlen=ventana)
        self._horas = deque(maxlen=ventana)
        self._n_transacciones = 0
        self._ultima_transaccion = 0.0

    def actualizar(self, tx: Transaccion):
        self._montos.append(tx.monto)
        self._paises.append(tx.pais)
        self._categorias.append(tx.categoria)
        self._horas.append(tx.hora)
        self._ultima_transaccion = tx.timestamp
        self._n_transacciones += 1

    def extraer_features(self, tx: Transaccion) -> Dict[str, float]:
        if len(self._montos) == 0:
            return {
                'monto_zscore': 0.0, 'monto_ratio': 1.0,
                'pais_nuevo': 1.0, 'categoria_nueva': 1.0,
                'hora_inusual': 0.0, 'tiempo_desde_ultima': 9999.0,
                'n_tx_historico': 0.0
            }

        montos_arr = np.array(self._montos)
        media_monto = montos_arr.mean()
        std_monto = montos_arr.std() + 1e-8
        horas_arr = np.array(self._horas)

        return {
            'monto_zscore': (tx.monto - media_monto) / std_monto,
            'monto_ratio': tx.monto / (media_monto + 1e-8),
            'pais_nuevo': float(tx.pais not in set(self._paises)),
            'categoria_nueva': float(tx.categoria not in set(self._categorias)),
            'hora_inusual': float(
                tx.hora < np.percentile(horas_arr, 10) or
                tx.hora > np.percentile(horas_arr, 90)
            ),
            'tiempo_desde_ultima': tx.timestamp - self._ultima_transaccion,
            'n_tx_historico': float(self._n_transacciones)
        }


class DetectorFraude:
    """Motor de scoring de fraude en tiempo real."""

    PESOS = np.array([0.35, 0.20, 0.15, 0.10, 0.10, -0.05, -0.05])
    FEATURES = ['monto_zscore', 'monto_ratio', 'pais_nuevo',
                'categoria_nueva', 'hora_inusual',
                'tiempo_desde_ultima', 'n_tx_historico']

    def __init__(self, umbral: float = 0.65):
        self.umbral = umbral
        self._perfiles: Dict[str, PerfilUsuario] = {}
        self._alertas: List[dict] = []

    def _sigmoid(self, x):
        return 1 / (1 + np.exp(-np.clip(x, -10, 10)))

    def procesar(self, tx: Transaccion) -> Tuple[float, bool]:
        if tx.usuario_id not in self._perfiles:
            self._perfiles[tx.usuario_id] = PerfilUsuario(tx.usuario_id)
        perfil = self._perfiles[tx.usuario_id]
        features = perfil.extraer_features(tx)
        x = np.array([features[f] for f in self.FEATURES])
        score = float(self._sigmoid(np.dot(x, self.PESOS)))
        es_fraude = score > self.umbral
        if es_fraude:
            self._alertas.append({
                'tx_id': tx.id, 'usuario': tx.usuario_id,
                'score': round(score, 4), 'monto': tx.monto
            })
        perfil.actualizar(tx)
        return score, es_fraude

    @property
    def alertas(self):
        return self._alertas


# Simulacion
np.random.seed(42)
detector = DetectorFraude(umbral=0.65)
paises = ['MX', 'US', 'CO', 'AR', 'BR']
categorias = ['supermercado', 'restaurante', 'gasolina', 'online', 'farmacia']

transacciones = []
for i in range(200):
    anomala = (i % 10 == 0)
    transacciones.append(Transaccion(
        id=f'TX{i:05d}',
        usuario_id=f'USR{i % 20:03d}',
        monto=float(np.random.uniform(5000, 50000) if anomala else np.random.uniform(50, 500)),
        pais=np.random.choice(['RU', 'NG']) if anomala else np.random.choice(paises),
        categoria=np.random.choice(categorias),
        hora=int(np.random.randint(2, 5) if anomala else np.random.randint(8, 22)),
        es_online=anomala,
        timestamp=time.time() + i * 60
    ))

t0 = time.time()
resultados = [detector.procesar(tx) for tx in transacciones]
tiempo_total = time.time() - t0

scores = [r[0] for r in resultados]
n_alertas = sum(1 for r in resultados if r[1])

print('Deteccion de Fraude — Resultados:')
print(f'  Transacciones procesadas : {len(transacciones):,}')
print(f'  Tiempo total             : {tiempo_total*1000:.1f} ms')
print(f'  Tiempo promedio/tx       : {tiempo_total/len(transacciones)*1000:.3f} ms')
print(f'  Alertas generadas        : {n_alertas} ({n_alertas/len(transacciones):.1%})')
print(f'  Score promedio           : {np.mean(scores):.4f}')
print('\nTop alertas de fraude:')
for a in detector.alertas[:5]:
    print(f"  {a['tx_id']} | Score: {a['score']:.4f} | Monto: ${a['monto']:,.0f} | {a['usuario']}")

---
## 🏥 Caso 2: Salud — Predicción de Readmisión Hospitalaria

**Contexto:** Hospital universitario predice qué pacientes tienen riesgo de readmisión en 30 días para intervención preventiva.

**Técnicas:** NumPy broadcasting, estandarización, SVD/PCA manual, regresión logística desde cero.

In [ ]:
import numpy as np

np.random.seed(2024)
n_pacientes = 1000

FEATURE_NAMES = [
    'edad', 'dias_hosp', 'n_comorbilidades', 'n_medicamentos',
    'presion_sistolica', 'hemoglobina', 'creatinina',
    'albumina', 'visitas_urgencias', 'charlson_idx'
]

# Grupos de riesgo
bajo_riesgo = np.random.multivariate_normal(
    mean=[55, 4, 1, 3, 125, 13, 0.9, 3.8, 0, 1],
    cov=np.diag([100, 4, 0.5, 2, 200, 4, 0.1, 0.3, 0.5, 1]),
    size=600
)
alto_riesgo = np.random.multivariate_normal(
    mean=[72, 10, 4, 8, 155, 10, 2.1, 2.8, 3, 5],
    cov=np.diag([150, 9, 2, 4, 300, 6, 0.5, 0.4, 1, 2]),
    size=400
)

X = np.vstack([bajo_riesgo, alto_riesgo])
y = np.hstack([np.zeros(600), np.ones(400)])

# Shuffle
idx = np.random.permutation(n_pacientes)
X, y = X[idx], y[idx]

# Estandarizacion manual con broadcasting
mu = X.mean(axis=0)           # shape (10,)
sigma = X.std(axis=0) + 1e-8  # shape (10,)
X_std = (X - mu) / sigma      # broadcasting: (1000,10) - (10,)

print('Dataset clinico:')
print(f'  Shape: {X.shape} | Readmisiones: {y.mean():.1%}')
print(f'  Media por feature (estandarizado): {X_std.mean(axis=0).round(3)}')
print(f'  Std  por feature (estandarizado): {X_std.std(axis=0).round(3)}')

In [ ]:
# PCA manual con SVD
U, s, Vt = np.linalg.svd(X_std, full_matrices=False)

varianza_explicada = (s ** 2) / np.sum(s ** 2)
varianza_acumulada = np.cumsum(varianza_explicada)

print('PCA — Varianza explicada por componente:')
for i in range(10):
    barra = '#' * int(varianza_explicada[i] * 50)
    print(f'  PC{i+1:2d}: {varianza_explicada[i]:.3f} ({varianza_acumulada[i]:.1%} acum) {barra}')

# Proyeccion en primeras 2 componentes
k = 2
X_pca = X_std @ Vt[:k].T  # shape (1000, 2)

# Separabilidad: distancia entre centroides de clases
centroide_0 = X_pca[y == 0].mean(axis=0)
centroide_1 = X_pca[y == 1].mean(axis=0)
distancia = np.linalg.norm(centroide_1 - centroide_0)

print(f'\nProyeccion PCA (k={k}):')
print(f'  Centroide clase 0 (sin readmision): {centroide_0.round(3)}')
print(f'  Centroide clase 1 (con readmision): {centroide_1.round(3)}')
print(f'  Distancia entre clases: {distancia:.4f}')
print(f'  Componentes top de PC1: {dict(zip(FEATURE_NAMES, Vt[0].round(3)))}')

In [ ]:
# Regresion Logistica desde cero con NumPy
def sigmoid(z):
    return 1 / (1 + np.exp(-np.clip(z, -20, 20)))

def entrenar_regresion_logistica(X, y, lr=0.01, epochs=500):
    n, p = X.shape
    X_b = np.column_stack([np.ones(n), X])  # bias
    w = np.zeros(p + 1)
    historial_loss = []

    for epoch in range(epochs):
        y_hat = sigmoid(X_b @ w)
        loss = -np.mean(y * np.log(y_hat + 1e-8) + (1 - y) * np.log(1 - y_hat + 1e-8))
        grad = X_b.T @ (y_hat - y) / n
        w -= lr * grad
        if epoch % 50 == 0:
            historial_loss.append((epoch, round(loss, 6)))

    return w, historial_loss

# Train/test split manual
split = int(0.8 * n_pacientes)
X_train, X_test = X_std[:split], X_std[split:]
y_train, y_test = y[:split], y[split:]

w, historial = entrenar_regresion_logistica(X_train, y_train, lr=0.05, epochs=500)

# Evaluacion
X_test_b = np.column_stack([np.ones(len(X_test)), X_test])
y_prob = sigmoid(X_test_b @ w)
y_pred = (y_prob > 0.5).astype(int)

accuracy = (y_pred == y_test).mean()
tp = ((y_pred == 1) & (y_test == 1)).sum()
fp = ((y_pred == 1) & (y_test == 0)).sum()
fn = ((y_pred == 0) & (y_test == 1)).sum()
precision = tp / (tp + fp + 1e-8)
recall    = tp / (tp + fn + 1e-8)
f1 = 2 * precision * recall / (precision + recall + 1e-8)

print('Modelo de Readmision Hospitalaria:')
print(f'  Accuracy  : {accuracy:.4f}')
print(f'  Precision : {precision:.4f}')
print(f'  Recall    : {recall:.4f}  <- critico en salud (minimizar falsos negativos)')
print(f'  F1-Score  : {f1:.4f}')
print('\nHistorial de entrenamiento (loss):')
for ep, loss in historial:
    print(f'  Epoch {ep:3d}: loss={loss}')
print('\nFeatures mas relevantes (|coeficiente|):')
nombres = ['bias'] + FEATURE_NAMES
importancia = sorted(zip(nombres, np.abs(w)), key=lambda x: x[1], reverse=True)
for nombre, coef in importancia[:5]:
    print(f'  {nombre:25s}: {coef:.4f}')

---
## 🛒 Caso 3: Retail — Sistema de Recomendación Colaborativo

**Contexto:** E-commerce con 500 usuarios y 200 productos. Se necesita recomendar productos no vistos usando filtrado colaborativo basado en SVD.

**Técnicas:** Matrices dispersas, SVD truncado, broadcasting para similaridad coseno.

In [ ]:
import numpy as np

np.random.seed(42)
n_usuarios = 200
n_productos = 100
n_factores_latentes = 8

# Factores latentes reales (generos: electronica, ropa, hogar, deportes...)
U_real = np.random.randn(n_usuarios, n_factores_latentes)
V_real = np.random.randn(n_factores_latentes, n_productos)
R_real = np.clip(U_real @ V_real + 3.5, 1, 5)

# Sparse: solo 15% de ratings observados
mascara = np.random.rand(n_usuarios, n_productos) < 0.15
R_obs = np.where(mascara, R_real + np.random.randn(n_usuarios, n_productos) * 0.3, 0)

print('Matriz de Ratings (Retail):')
print(f'  Shape: {R_obs.shape}')
print(f'  Ratings observados: {mascara.sum():,} ({mascara.mean():.1%})')
print(f'  Rating promedio: {R_obs[mascara].mean():.2f}')

In [ ]:
# SVD Truncado para factorizacion
U, s, Vt = np.linalg.svd(R_obs, full_matrices=False)

print('Reconstruccion con k factores:')
print(f'{"k":>4} | {"Varianza":>10} | {"RMSE":>8}')
print('-' * 28)
varianza_acum = np.cumsum(s**2) / np.sum(s**2)

for k in [1, 2, 4, 8, 16, 32]:
    R_k = U[:, :k] @ np.diag(s[:k]) @ Vt[:k, :]
    rmse = np.sqrt(np.mean((R_real[mascara] - R_k[mascara])**2))
    print(f'{k:4d} | {varianza_acum[k-1]:10.2%} | {rmse:8.4f}')

# Modelo optimo con k=8
k_opt = 8
R_pred = U[:, :k_opt] @ np.diag(s[:k_opt]) @ Vt[:k_opt, :]
R_pred = np.clip(R_pred, 1, 5)

# Recomendaciones para usuario 0
usuario_id = 0
ratings_usuario = R_obs[usuario_id]
productos_vistos = set(np.where(ratings_usuario > 0)[0])

scores_pred = R_pred[usuario_id]
recomendaciones = [
    (p, round(scores_pred[p], 3))
    for p in np.argsort(scores_pred)[::-1]
    if p not in productos_vistos
][:5]

print(f'\nRecomendaciones para usuario {usuario_id}:')
print(f'  Productos ya vistos: {len(productos_vistos)}')
print('  Top 5 recomendaciones (producto_id, score_predicho):')
for prod_id, score in recomendaciones:
    print(f'    Producto {prod_id:3d}: score={score:.3f}')

In [ ]:
# Similaridad coseno entre usuarios (para filtrado colaborativo basado en usuarios)
# Usando broadcasting — sin loops
U_emb = U[:, :k_opt] * s[:k_opt]  # embeddings de usuarios

# Normas de cada usuario
normas = np.linalg.norm(U_emb, axis=1, keepdims=True)  # shape (200, 1)
U_norm = U_emb / (normas + 1e-8)                        # normalizar

# Matriz de similaridad: (200, 200) — broadcasting
sim_matrix = U_norm @ U_norm.T

# Usuarios mas similares al usuario 0
similitudes_u0 = sim_matrix[0]
similitudes_u0[0] = -1  # excluir el propio
top_similares = np.argsort(similitudes_u0)[::-1][:5]

print('Usuarios mas similares al usuario 0:')
for u in top_similares:
    print(f'  Usuario {u:3d}: similitud coseno = {sim_matrix[0, u]:.4f}')

---
## 🏭 Caso 4: Industria — Mantenimiento Predictivo de Maquinaria

**Contexto:** Planta manufacturera con 50 maquinas generando lecturas cada segundo durante 24h. El dataset es demasiado grande para RAM.

**Técnicas:** Generators para procesamiento lazy, numpy memmap, estadisticas en ventana deslizante.

In [ ]:
import numpy as np
import tempfile
import os
from collections import deque

np.random.seed(42)

# Simular stream de sensores con generator
def stream_sensores(n_maquinas: int, n_segundos: int, falla_prob: float = 0.01):
    """Generator: simula lecturas en tiempo real de sensores industriales."""
    for t in range(n_segundos):
        for m in range(n_maquinas):
            # Falla simulada: incremento de temperatura + vibracion
            en_falla = np.random.random() < falla_prob
            yield {
                'timestamp': t,
                'maquina_id': m,
                'temperatura': float(
                    np.random.normal(85, 5) + (30 if en_falla else 0)
                ),
                'vibracion': float(
                    np.random.normal(2.5, 0.5) + (5 if en_falla else 0)
                ),
                'presion': float(np.random.normal(120, 10)),
                'rpm': float(
                    np.random.normal(3000, 100) - (500 if en_falla else 0)
                ),
                'en_falla': en_falla
            }


def procesar_en_batches(stream, batch_size: int):
    """Wrapper generator: agrupa lecturas en mini-batches."""
    batch = []
    for lectura in stream:
        batch.append(lectura)
        if len(batch) == batch_size:
            yield batch
            batch = []
    if batch:
        yield batch


class DetectorAnomalias:
    """Detector de anomalias en streaming con ventana deslizante."""

    def __init__(self, n_maquinas: int, ventana: int = 60):
        self.n_maquinas = n_maquinas
        self.ventana = ventana
        # Una deque por maquina por sensor
        self._historiales = {
            m: {
                'temperatura': deque(maxlen=ventana),
                'vibracion': deque(maxlen=ventana),
            }
            for m in range(n_maquinas)
        }
        self.alertas = []
        self.n_procesadas = 0

    def procesar_batch(self, batch: list) -> list:
        nuevas_alertas = []
        for lectura in batch:
            m = lectura['maquina_id']
            hist = self._historiales[m]

            for sensor in ['temperatura', 'vibracion']:
                hist[sensor].append(lectura[sensor])

            if len(hist['temperatura']) >= 10:
                temp_arr = np.array(hist['temperatura'])
                vib_arr = np.array(hist['vibracion'])
                z_temp = (lectura['temperatura'] - temp_arr.mean()) / (temp_arr.std() + 1e-8)
                z_vib  = (lectura['vibracion'] - vib_arr.mean()) / (vib_arr.std() + 1e-8)

                if abs(z_temp) > 3 or abs(z_vib) > 3:
                    alerta = {
                        'timestamp': lectura['timestamp'],
                        'maquina_id': m,
                        'z_temp': round(z_temp, 2),
                        'z_vib': round(z_vib, 2),
                        'real_falla': lectura['en_falla']
                    }
                    nuevas_alertas.append(alerta)
                    self.alertas.append(alerta)

            self.n_procesadas += 1
        return nuevas_alertas


# Ejecutar pipeline
N_MAQUINAS = 20
N_SEGUNDOS = 500
BATCH_SIZE = 100

detector = DetectorAnomalias(n_maquinas=N_MAQUINAS)
stream = stream_sensores(N_MAQUINAS, N_SEGUNDOS, falla_prob=0.02)

import time
t0 = time.time()
for batch in procesar_en_batches(stream, BATCH_SIZE):
    detector.procesar_batch(batch)

elapsed = time.time() - t0
total_lecturas = N_MAQUINAS * N_SEGUNDOS

n_verdaderas = sum(1 for a in detector.alertas if a['real_falla'])
n_falsas     = sum(1 for a in detector.alertas if not a['real_falla'])

print('Mantenimiento Predictivo — Resultados:')
print(f'  Lecturas procesadas  : {total_lecturas:,}')
print(f'  Tiempo de proceso    : {elapsed:.3f}s')
print(f'  Throughput           : {total_lecturas/elapsed:,.0f} lecturas/seg')
print(f'  Total alertas        : {len(detector.alertas)}')
print(f'  Verdaderas positivas : {n_verdaderas}')
print(f'  Falsas positivas     : {n_falsas}')
if len(detector.alertas) > 0:
    print(f'  Precision            : {n_verdaderas/len(detector.alertas):.2%}')
print('\nPrimeras 3 alertas:')
for a in detector.alertas[:3]:
    print(f"  t={a['timestamp']:4d} | Maquina {a['maquina_id']:2d} | "
          f"z_temp={a['z_temp']:5.2f} | z_vib={a['z_vib']:5.2f} | "
          f"{'FALLA REAL' if a['real_falla'] else 'Falsa alarma'}")

---
## 🌱 Caso 5: Agricultura — Clasificación de Enfermedades en Cultivos

**Contexto:** Cooperativa agroindustrial necesita clasificar enfermedades en plantas a partir de mediciones de sensores de campo (NDVI, temperatura foliar, humedad, etc.).

**Técnicas:** Pipeline OOP completo (BaseTransformer), feature engineering, clasificador KNN desde cero con NumPy.

In [ ]:
import numpy as np
from abc import ABC, abstractmethod
from typing import Dict, Any, List


# ── Base ─────────────────────────────────────────────────────────────────
class BaseTransformer(ABC):
    def __init__(self):
        self._fitted = False

    @abstractmethod
    def fit(self, X, y=None):
        ...

    @abstractmethod
    def transform(self, X):
        ...

    def fit_transform(self, X, y=None):
        return self.fit(X, y).transform(X)

    def _check(self):
        if not self._fitted:
            raise RuntimeError(f'{self.__class__.__name__} no ha sido entrenado')

    def __repr__(self):
        return f'{self.__class__.__name__}(fitted={self._fitted})'


# ── Transformadores ───────────────────────────────────────────────────────
class MinMaxScaler(BaseTransformer):
    def fit(self, X, y=None):
        X = np.asarray(X, dtype=np.float64)
        self._min = X.min(axis=0)
        self._max = X.max(axis=0)
        self._range = self._max - self._min
        self._range[self._range == 0] = 1
        self._fitted = True
        return self

    def transform(self, X):
        self._check()
        return (np.asarray(X, dtype=np.float64) - self._min) / self._range


class PolinomialFeatures(BaseTransformer):
    """Agrega features polinomiales de grado 2 (interacciones)."""

    def __init__(self, interaction_only: bool = False):
        super().__init__()
        self.interaction_only = interaction_only
        self._n_features = None

    def fit(self, X, y=None):
        self._n_features = X.shape[1]
        self._fitted = True
        return self

    def transform(self, X):
        self._check()
        X = np.asarray(X, dtype=np.float64)
        partes = [X]
        n = X.shape[1]
        for i in range(n):
            for j in range(i if not self.interaction_only else i + 1, n):
                partes.append((X[:, i] * X[:, j]).reshape(-1, 1))
        return np.hstack(partes)


class Pipeline:
    """Pipeline secuencial de transformadores."""

    def __init__(self, steps: List[tuple]):
        self.steps = steps
        self._fitted = False

    def fit(self, X, y=None):
        X_curr = X
        for name, t in self.steps:
            X_curr = t.fit_transform(X_curr, y)
        self._fitted = True
        return self

    def transform(self, X):
        X_curr = X
        for _, t in self.steps:
            X_curr = t.transform(X_curr)
        return X_curr

    def fit_transform(self, X, y=None):
        return self.fit(X, y).transform(X)

    def __repr__(self):
        pasos = ', '.join(f"{n}:{t.__class__.__name__}" for n, t in self.steps)
        return f'Pipeline([{pasos}])'


# ── Clasificador KNN desde cero ───────────────────────────────────────────
class KNNClassifier:
    """K-Nearest Neighbors vectorizado con NumPy."""

    def __init__(self, k: int = 5):
        self.k = k
        self._X_train = None
        self._y_train = None

    def fit(self, X, y):
        self._X_train = np.asarray(X, dtype=np.float64)
        self._y_train = np.asarray(y)
        return self

    def predict(self, X):
        X = np.asarray(X, dtype=np.float64)
        # Distancias euclidianas: broadcasting (n_test, n_train)
        diff = X[:, np.newaxis, :] - self._X_train[np.newaxis, :, :]
        distancias = np.sqrt((diff ** 2).sum(axis=2))
        vecinos = np.argsort(distancias, axis=1)[:, :self.k]
        etiquetas_vecinos = self._y_train[vecinos]
        # Votacion por mayoria
        predicciones = []
        for fila in etiquetas_vecinos:
            valores, conteos = np.unique(fila, return_counts=True)
            predicciones.append(valores[np.argmax(conteos)])
        return np.array(predicciones)


# ── Dataset: sensores agronomicos ─────────────────────────────────────────
np.random.seed(99)
CLASES = ['sana', 'mildiu', 'roya', 'marchitez']
n_por_clase = 150

medias = {
    'sana':       [0.75, 25, 65, 0.8, 320],
    'mildiu':     [0.55, 28, 80, 0.6, 280],
    'roya':       [0.50, 30, 70, 0.5, 260],
    'marchitez':  [0.35, 35, 45, 0.4, 240]
}
FEATURE_NAMES = ['NDVI', 'temp_foliar', 'humedad', 'clorofila', 'luz_PAR']

X_list, y_list = [], []
for idx_c, clase in enumerate(CLASES):
    muestras = np.random.randn(n_por_clase, 5) * np.array([0.05, 2, 5, 0.05, 15])
    muestras += np.array(medias[clase])
    X_list.append(muestras)
    y_list.extend([idx_c] * n_por_clase)

X_agro = np.vstack(X_list)
y_agro = np.array(y_list)

# Shuffle
idx = np.random.permutation(len(y_agro))
X_agro, y_agro = X_agro[idx], y_agro[idx]

print('Dataset Agricola:')
print(f'  Shape: {X_agro.shape}')
for i, c in enumerate(CLASES):
    print(f'  Clase {i} ({c}): {(y_agro==i).sum()} muestras')

In [ ]:
# Pipeline completo + KNN
split = int(0.8 * len(y_agro))
X_tr, X_te = X_agro[:split], X_agro[split:]
y_tr, y_te = y_agro[:split], y_agro[split:]

pipe = Pipeline([
    ('minmax', MinMaxScaler()),
    ('poly',   PolinomialFeatures(interaction_only=True))
])

X_tr_proc = pipe.fit_transform(X_tr)
X_te_proc = pipe.transform(X_te)

print(f'Shape original : {X_tr.shape}')
print(f'Shape procesado: {X_tr_proc.shape} (features polinomiales incluidas)')
print(f'\nPipeline: {pipe}')

# Evaluar distintos valores de k
print('\nEvaluacion KNN con distintos k:')
print(f'  {"k":>3} | {"Accuracy":>10}')
print('  ' + '-' * 18)
mejor_k, mejor_acc = 1, 0
for k in [1, 3, 5, 7, 11, 15]:
    knn = KNNClassifier(k=k)
    knn.fit(X_tr_proc, y_tr)
    y_pred = knn.predict(X_te_proc)
    acc = (y_pred == y_te).mean()
    marker = ' <- mejor' if acc > mejor_acc else ''
    print(f'  {k:3d} | {acc:10.4f}{marker}')
    if acc > mejor_acc:
        mejor_acc, mejor_k = acc, k

# Matriz de confusion con mejor k
knn_final = KNNClassifier(k=mejor_k)
knn_final.fit(X_tr_proc, y_tr)
y_pred_final = knn_final.predict(X_te_proc)

n_clases = len(CLASES)
conf_matrix = np.zeros((n_clases, n_clases), dtype=int)
for real, pred in zip(y_te, y_pred_final):
    conf_matrix[real, pred] += 1

print(f'\nMatriz de Confusion (k={mejor_k}):')
header = f'{"":10}' + ''.join(f'{c:12}' for c in CLASES)
print('  ' + header)
for i, clase in enumerate(CLASES):
    fila = ''.join(f'{conf_matrix[i,j]:12}' for j in range(n_clases))
    print(f'  {clase:10}{fila}')

# Precision por clase
print('\nMetricas por clase:')
for i, clase in enumerate(CLASES):
    tp = conf_matrix[i, i]
    fp = conf_matrix[:, i].sum() - tp
    fn = conf_matrix[i, :].sum() - tp
    prec = tp / (tp + fp + 1e-8)
    rec  = tp / (tp + fn + 1e-8)
    f1   = 2 * prec * rec / (prec + rec + 1e-8)
    print(f'  {clase:12}: precision={prec:.3f} recall={rec:.3f} f1={f1:.3f}')

---
## 🧠 Resumen del Módulo 1

### Lo aprendido en los 5 casos reales:

| Industria | Tecnica clave | Concepto Python |
|-----------|---------------|-----------------|
| Fintech (fraude) | Scoring en tiempo real | `__slots__`, `deque`, OOP |
| Salud (readmision) | Regresion logistica manual | NumPy broadcasting, SVD |
| Retail (recomendacion) | Factorizacion matricial | SVD truncado, similaridad coseno |
| Industria (mantenimiento) | Deteccion anomalias streaming | Generators, ventana deslizante |
| Agro (enfermedades) | Clasificacion multiclase | Pipeline OOP, KNN vectorizado |

### Principios del Modulo 1:
```
1. Estructura adecuada  -> elegir list/dict/set/tuple segun el acceso
2. Vectorizacion        -> nunca loops si NumPy puede hacerlo
3. OOP con patron fit/transform -> reproducibilidad y evitar data leakage
4. Memoria eficiente    -> float32, __slots__, generators, memmap
5. Validacion siempre   -> check_fitted, validar shapes, manejar NaN
```

**Siguiente modulo:** Pandas avanzado, Exploratory Data Analysis y Feature Engineering.